In [0]:
from pyspark.sql import functions as F

T_trans = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_sessions = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_session")
T_locations = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")
T_films = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_film")
T_ticket = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_ticket_type")
T_date = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_date")
T_items = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_item")
T_genre = spark.table("cpx_dataanalytics_gold.customer360.film_composite_genres")
T_subscriber = spark.table("cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot")
T_events = spark.table("cpx_dataanalytics_gold.marketing.mkt_subscription_events")
T_emails = spark.table("cpx_dataanalytics_gold.marketing.cpx_d_customer_email_address")
T_rfm = spark.table("cpx_dataanalytics_gold.customer360.c360_f_rfm_model")
T_customer = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_customer")

In [0]:
# Clears all cached Spark DataFrames in the session
spark.catalog.clearCache()

In [0]:
Total_Population = (
    T_trans
    .join(T_date.select("dateid", "datefull"), T_trans.VisitDateId == T_date.dateid, "left")
    .join(T_locations.select("LocationID", "LC_Location_Type_Description"), T_trans.LocationId == T_locations.LocationID, "left")
    .filter(F.col("LC_Location_Type_Description") == "Cineplex Theatre")
    .filter(F.col("TicketFLAG")== "Yes")
    .withColumn("Year", F.year(F.col("datefull")))
    .groupBy("Year")
    .agg(
        F.sum("Quantity").alias("Total_Tickets")
    )
)

In [0]:
Total_Population.display()

In [0]:
pip install openpyxl

In [0]:
import os

folder = "/Workspace/Users/ranjan.mishra@cineplex.com/CPX-Marketing-Enablement-Analytics/CPX-CineClub/Monthly Reports"

for root, dirs, files in os.walk(folder):
    print(f"📁 {root}")
    for f in files:
        print(f"   📄 {f}")

In [0]:
import pandas as pd

pdf = pd.read_excel(
    "/Workspace/Users/ranjan.mishra@cineplex.com/CPX-Marketing-Enablement-Analytics/CPX-CineClub/Monthly Reports/February 2026/MOM_February_2026.xlsx",
    usecols=[
        "customers.id",
        "subscriptions.id",
        "subscriptions.plan_id",
        "subscriptions.status",
        "subscriptions.current_term_start",
        "subscriptions.current_term_end",
        "subscriptions.created_at",
        "subscriptions.cancelled_at",
    ],
)

# 3) Parse datetime columns in pandas and convert to a consistent string format (Arrow-safe)
dt_cols = [
    "subscriptions.current_term_start",
    "subscriptions.current_term_end",
    "subscriptions.created_at",
    "subscriptions.cancelled_at",
]

for c in dt_cols:
    pdf[c] = pd.to_datetime(pdf[c], format="%d-%b-%Y %H:%M", errors="coerce")
    pdf[c] = pdf[c].dt.strftime("%Y-%m-%d %H:%M:%S")  # NaT -> NaN

# 4) Ensure ID/status fields are strings (prevents mixed-type Arrow issues)
for c in ["customers.id", "subscriptions.id", "subscriptions.plan_id", "subscriptions.status"]:
    pdf[c] = pdf[c].astype("string")


# 5) Convert pandas -> Spark DataFrame (remote cluster)
df = spark.createDataFrame(pdf)

# 6) Project ONLY the final desired columns (no repetition):
#    - Rename dotted columns to clean names
#    - Parse the datetime strings into Spark timestamps
df_main = (
    df
    .select(
        F.col("`customers.id`").alias("customer_id"),
        F.col("`subscriptions.id`").alias("subscription_id"),
        F.col("`subscriptions.plan_id`").alias("plan_id"),
        F.col("`subscriptions.status`").alias("subscription_status"),
        F.to_timestamp(F.col("`subscriptions.current_term_start`"), "yyyy-MM-dd HH:mm:ss").alias("current_term_start"),
        F.to_timestamp(F.col("`subscriptions.current_term_end`"),   "yyyy-MM-dd HH:mm:ss").alias("current_term_end"),
        F.to_timestamp(F.col("`subscriptions.created_at`"),         "yyyy-MM-dd HH:mm:ss").alias("created_at"),
        F.to_timestamp(F.col("`subscriptions.cancelled_at`"),       "yyyy-MM-dd HH:mm:ss").alias("cancelled_at"),
    )
    .dropDuplicates([
        "customer_id",
        "plan_id",
        "subscription_status",
        "current_term_start",
        "current_term_end",
        "created_at",
        "cancelled_at"
    ])
    .withColumn("created_at_year_month", F.date_format(F.col("created_at"), "yyyy-MM"))
    .withColumn("current_term_end_year_month", F.date_format(F.col("current_term_end"), "yyyy-MM"))
    .withColumn("cancelled_at_year_month", F.date_format(F.col("cancelled_at"), "yyyy-MM"))
    .withColumn(
        "days_in_program",
        F.when(
            F.col("cancelled_at").isNull() | F.col("created_at").isNull(),
            F.lit(-9999)
        ).otherwise(F.datediff(F.col("cancelled_at"), F.col("created_at"))).cast("int")
    )
)